# 07 · January 2026 acquisition forecast

**Method:** seasonal index × current run-rate. With only three seasonal cycles, ARIMA / Holt-Winters can't be fitted reliably, so a transparent index method is used.

- Seasonal index per ISO week = 3-year mean for that week ÷ grand weekly mean (notebook 02).
- Level = trailing-26-week **deseasonalised** mean of new customers per week.
- Forecast = level × January W1–W4 indices.

Supports deck slides **8, A4**.

![Forecast](../reports/figures/s8_forecast.png)

In [1]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
from common import load, customers, weekly_new, CHANNEL_ORDER, LAST_WEEK_IDX
pd.set_option("display.width", 140)
df = load()

In [2]:
w = weekly_new(df).values
iso = np.tile(np.arange(1, 53), 3)
idx = np.array([w[iso == k].mean() / w.mean() for k in range(1, 53)])
jan = idx[:4]
level = (w / idx[iso - 1])[-26:].mean()
fc = level * jan
print(f"January indices W1–W4: {np.round(jan, 2)}")
print(f"Level (trailing 26 wks, deseasonalised): {level:.1f} new customers/wk")
print(f"Forecast by week: {np.round(fc, 1)}  ->  total {fc.sum():.0f} new customers in weeks 157–160")

January indices W1–W4: [1.21 1.18 1.05 1.11]
Level (trailing 26 wks, deseasonalised): 58.1 new customers/wk
Forecast by week: [70.3 68.3 61.  64.7]  ->  total 264 new customers in weeks 157–160


In [3]:
full_2025 = w[104:].mean()
print(f"Cross-check: full-2025 raw mean {full_2025:.1f}/wk -> {full_2025 * jan.sum():.0f} over the same 4 weeks")
print("Actual first-4-week January totals:", {y: int(w[i * 52:i * 52 + 4].sum()) for i, y in enumerate((2023, 2024, 2025))})

Cross-check: full-2025 raw mean 56.6/wk -> 257 over the same 4 weeks
Actual first-4-week January totals: {2023: 97, 2024: 404, 2025: 292}


## Planning range

The deck presents **~264** with a planning range of **~230–300**. As stated on slide A4, the range is deliberately widened beyond the spread of the point methods because there are only three seasonal cycles of history. It is a judgement-based planning range, not a statistical confidence interval, and is set as an explicit input (`PLANNING_RANGE`) in `src/make_charts.py`.